# 17 – Ingestion Pipeline (Chunking & Document Loading)

The ingestion pipeline processes governance documents into the vector store:
1. **Loaders** — PDF, DOCX, PPTX, XLSX, TXT, MD
2. **Chunker** — splits documents, infers product tags, adds SHA-256 hashes
3. **Embedder** — calls OpenAI embeddings (skipped in this notebook)
4. **Store** — pgvector upsert with deduplication (skipped in this notebook)

This notebook focuses on the chunker logic which has no external dependencies.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

In [ ]:
from langchain_core.documents import Document
from ingestion.chunker import chunk_documents, _infer_product

## 1. Product inference from filename

In [ ]:
filenames = [
    'retention_policy_v2.pdf',
    'bookings_governance_standard.docx',
    'cac_calculation_guide.pdf',
    'customer_ltv_framework.pdf',
    'data_governance_framework.pdf',
    'incident_response_playbook.pdf',
    'revenue_recognition.docx',
]

for f in filenames:
    product = _infer_product(f)
    print(f'{f:<45} => {product}')

## 2. Chunking a single document

In [ ]:
doc = Document(
    page_content=(
        "Data Governance Framework\n\n"
        "All data assets must be registered in Collibra within 30 days of creation. "
        "Data owners are accountable for maintaining quality scores above 90%. "
        "Quarterly reviews are mandatory for all tier-1 data products.\n\n"
        "Retention Policy\n\n"
        "Customer retention metrics must maintain a GRR above 85%. "
        "Churn above 15% triggers automatic escalation to Customer Success. "
        "Monthly reporting is required for all retention KPIs."
    ),
    metadata={'source': 'retention_policy.pdf'},
)

chunks = chunk_documents([doc], chunk_size=200, chunk_overlap=20)

print(f'Input: 1 document, {len(doc.page_content)} chars')
print(f'Output: {len(chunks)} chunks\n')

for i, chunk in enumerate(chunks):
    print(f'Chunk {i}:')
    print(f'  content     : {chunk.page_content[:80]}...')
    print(f'  product     : {chunk.metadata["product"]}')
    print(f'  topic       : {chunk.metadata["topic"]}')
    print(f'  chunk_index : {chunk.metadata["chunk_index"]}')
    print(f'  content_hash: {chunk.metadata["content_hash"][:16]}...')
    print()

## 3. Chunk deduplication via content hash

In [ ]:
# Same document chunked twice — hashes should match
chunks1 = chunk_documents([doc], chunk_size=200, chunk_overlap=20)
chunks2 = chunk_documents([doc], chunk_size=200, chunk_overlap=20)

hashes1 = {c.metadata['content_hash'] for c in chunks1}
hashes2 = {c.metadata['content_hash'] for c in chunks2}

print(f'First chunking hashes : {len(hashes1)}')
print(f'Second chunking hashes: {len(hashes2)}')
print(f'Hashes match (dedup)  : {hashes1 == hashes2}')

## 4. Chunk multiple documents of different products

In [ ]:
docs = [
    Document(page_content='Retention GRR must stay above 85%. Churn escalation triggers at 15%.', metadata={'source': 'retention_policy.pdf'}),
    Document(page_content='Bookings must reconcile with CRM within 24 hours. ASC 606 applies.', metadata={'source': 'bookings_standard.pdf'}),
    Document(page_content='CAC blended should not exceed 36-month payback period per marketing guidelines.', metadata={'source': 'cac_rules.pdf'}),
    Document(page_content='LTV:CAC below 3x triggers Data Science review. 24-month cohort analysis required.', metadata={'source': 'customer_ltv_framework.pdf'}),
]

all_chunks = chunk_documents(docs, chunk_size=512, chunk_overlap=64)

from collections import Counter
product_counts = Counter(c.metadata['product'] for c in all_chunks)

print(f'Total chunks: {len(all_chunks)}')
print('By product:')
for product, count in sorted(product_counts.items()):
    print(f'  {product:<12}: {count} chunk(s)')

## 5. Chunk size tuning

In [ ]:
# Show how chunk size affects split count
long_doc = Document(
    page_content=' '.join([f'Sentence {i} about data governance standards.' for i in range(50)]),
    metadata={'source': 'governance_framework.pdf'},
)

for chunk_size in [128, 256, 512, 1024]:
    chunks = chunk_documents([long_doc], chunk_size=chunk_size, chunk_overlap=20)
    print(f'chunk_size={chunk_size:4d}  =>  {len(chunks)} chunks')

## 6. Supported document formats (loaders reference)

In [ ]:
# Show which extensions are supported by load_document()
# (importing without calling to avoid needing actual files)
from ingestion.loaders import load_document
import inspect

src = inspect.getsource(load_document)
print('Supported extensions in load_document():')
for line in src.split('\n'):
    if 'ext ==' in line or 'ext in' in line:
        print(' ', line.strip())